In [51]:
import numpy
from alternet.annotation import *
from alternet.data_preprocessing import standardize_dataframe
import numpy
import os
import os.path as op


from alternet.alternet_class import *
from alternet import postprocessing
from alternet.edge_categorization import *
from alternet.gtex_dataloader import *

In [ ]:


data_path = "/data/bionets/og86asub/alternet-project/alternet/data"
results_path = "/data/bionets/og86asub/alternet-project/alternet/raw_networks/"

# Reference files
appris_path = "appris_data.appris.txt"
digger_path = "digger_data.csv"
biomart_path = "biomart.txt"
tf_list_path = "allTFs_hg38.txt"
sf_list_path = "splicefactors.csv"

# Expression data
gtex_transcript_tpm_path = "GTEx_Analysis_v10_RSEMv1.3.3_transcripts_tpm.txt"
gtex_sample_attributes_path = "GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt"

# Tissue to analyze
TISSUE = "Heart"
CONDITION = TISSUE

# Number of GRNBoost2 runs
N_RUNS = 1

os.makedirs(results_path, exist_ok=True)



In [3]:
biomart = pd.read_csv(op.join(data_path, biomart_path), sep='\t')
tx2gene = dict(zip(biomart['Transcript stable ID'], biomart['Gene stable ID']))
gene2tx = biomart.groupby('Gene stable ID')['Transcript stable ID'].apply(set).to_dict()
appris_df = pd.read_csv(op.join(data_path,appris_path), sep='\t')
digger_df = pd.read_csv(op.join(data_path,digger_path), low_memory=False)
# Load and map TF list
tf_list_raw = pd.read_csv(op.join(data_path,tf_list_path), sep='\t', header=None)
tf_list = map_tf_ids(tf_list_raw, biomart)

In [ ]:


VARIANCE_PERCENTILE = 0.7  # Keep top 30%

In [ ]:
# Load and map SF list
sf_list_raw = pd.read_csv(op.join(data_path, sf_list_path), header=0, sep = ',')
sf_list = map_sf_ids(sf_list_raw.loc[:, ['Splicing_Factor']], biomart)
# Combine TF and SF lists
regulator_list = combine_tf_sf_lists(tf_list, sf_list)
tx_to_regtype = dict(zip(regulator_list['Transcript stable ID'], regulator_list['Regulator_type']))
gene_to_regtype = regulator_list.groupby('Gene stable ID')['Regulator_type'].first().to_dict()


In [8]:
gtex_data_dir = '/data/bionets/datasets/hackathon/data/GTEX'
params = {'sample_attributes': op.join(gtex_data_dir, 'GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt'), 'tissue': CONDITION, 'transcript_data':op.join(gtex_data_dir, 'GTEx_Analysis_2017-06-05_v8_RSEMv1.3.0_transcript_tpm.gct')}
tissue_ids = retrieve_GTEX_tissue_sampleids(params['sample_attributes'], tissue=params['tissue'])
transcript_data = read_GTEX_transcript_expression(params['transcript_data'], tissue_ids)
transcript_data = clean_GTEX_tissue_transcript_counts(transcript_data, biomart)
transcript_data = variance_filtering(transcript_data)


Retrieving tissue sample IDs
Reading Transcript expression data
Cleaning up counts


In [9]:
canonical_grn = pd.read_csv(op.join(results_path, f"{CONDITION}_canonical_raw.tsv"), sep='\t')

as_source_grn = pd.read_csv(op.join(results_path,  f"{CONDITION}_as_aware_source_raw.tsv"), sep='\t')

fully_as_grn= pd.read_csv(op.join(results_path, f"{CONDITION}_fully_as_aware_raw.tsv"), sep='\t')


In [52]:
alternet_obj21 = Alternet(canonical_grn, as_source_grn, fully_as_grn, transcript_data, regulator_list, tx_to_regtype , 'gene_id', 'transcript_id', min_frequency=10, importance_percentile = 0.9)

Computing set A

Set A: 207,543 rows

Category distribution:
  source_gene_specific             78,094 ( 37.6%)
  source_isoform_specific          47,802 ( 23.0%)
  source_equivalent                45,339 ( 21.8%)
  source_ambiguous                 36,308 ( 17.5%)
Set D diambiguation
Corrected version
Computing set D
Computing set B

Final Set B: 214,236 rows

Category distribution:
  target_isoform_specific          77,229 ( 36.0%)
  target_equivalent                58,303 ( 27.2%)
  target_gene_specific             39,535 ( 18.5%)
  target_ambiguous                 39,169 ( 18.3%)

By regulator type:
       source_transcript target_transcript      target_gene     S2_mean  \
132292   ENST00000405885   ENST00000437654  ENSG00000125347  264.621636   
167214   ENST00000494018   ENST00000441012  ENSG00000025434  260.696169   
105008   ENST00000376112   ENST00000376105  ENSG00000125968  252.692664   
199748   ENST00000578202   ENST00000360772  ENSG00000065534  248.828893   
75679    ENST00

In [ ]:
results_path = "/data/bionets/og86asub/alternet-project/alternet/results_gtex"
os.makedirs(op.join(results_path, CONDITION), exist_ok = True)
alternet_obj21.set_a.to_csv(op.join(results_path, CONDITION, f"{CONDITION}_set_a.tsv"), sep = '\t')
alternet_obj21.set_b.to_csv(op.join(results_path, CONDITION, f"{CONDITION}_set_b.tsv"), sep = '\t')
alternet_obj21.set_c.to_csv(op.join(results_path, CONDITION, f"{CONDITION}_set_c.tsv"), sep = '\t')
alternet_obj21.set_d.to_csv(op.join(results_path, CONDITION, f"{CONDITION}_set_d.tsv"), sep = '\t')

In [38]:
tfsf_sf_like = alternet_obj21.set_d_full[alternet_obj21.set_d_full['tfsf_category'] == 'tfsf_sf_like'].copy()
net3_sf = alternet_obj21.as_full[alternet_obj21.as_full['reg_type'] == 'SF'].copy()
udf = alternet_obj21.usage_df.drop(columns = {'gene_id'}).set_index("transcript_id")
t_temps = alternet_obj21.transcript_data.drop(columns = {alternet_obj21.gene_col}).set_index(alternet_obj21.transcript_col)
sf_edges = pd.concat([net3_sf, tfsf_sf_like])


In [ ]:
sf_edges.groupby(['source_transcript', 'source_gene', 'target_gene']).agg({
        'mean_importance': ['sum', 'max', 'count'],
        'median_importance': ['sum', 'max'],
        'target_transcript': lambda x: set(x)
    }).reset_index()

In [ ]:
compute_set_c(sf_edges[0:10], alternet_obj21.transcript_data, alternet_obj21.gene2tx, udf, alternet_obj21.reliability_df, alternet_obj21.sample_cols, epsilon=1e-6, n_cores=32)


In [ ]:
def score_targets(edges, target_col, importance_col='median_importance'):
    """
    Compute target scores as weighted in-degree.
    score(target) = sum of importance for all edges targeting it
    """
    scores = edges.groupby(target_col)[importance_col].sum()
    return scores.sort_values(ascending=False)


In [ ]:
def project_tx_to_gene(tx_scores, tx2gene, method='max'):
    """
    Project transcript-level scores to gene-level.
    
    Parameters:
    - tx_scores: pd.Series mapping transcript_id -> score
    - tx2gene: dict mapping transcript_id -> gene_id
    - method: 'max' (default) or 'sum'
    
    Returns:
    - gene_scores: pd.Series mapping gene_id -> score
    - rep_tx: dict mapping gene_id -> representative transcript
    """
    df = pd.DataFrame({
        'transcript_id': tx_scores.index,
        'score': tx_scores.values
    })
    df['gene_id'] = df['transcript_id'].map(tx2gene)
    df = df.dropna(subset=['gene_id'])
    
    if method == 'max':
        idx = df.groupby('gene_id')['score'].idxmax()
        result = df.loc[idx].set_index('gene_id')
        gene_scores = result['score'].sort_values(ascending=False)
        rep_tx = result['transcript_id'].to_dict()
    elif method == 'sum':
        gene_scores = df.groupby('gene_id')['score'].sum().sort_values(ascending=False)
        rep_tx = {}
    else:
        raise ValueError(f"Unknown method: {method}")
    
    return gene_scores, rep_tx


In [ ]:

tx2gene = dict(zip(biomart['Transcript stable ID'], biomart['Gene stable ID']))
gene2symbol = dict(zip(biomart['Gene stable ID'], biomart['Gene name']))
symbol2gene = dict(zip(biomart['Gene name'], biomart['Gene stable ID']))

In [ ]:
TOP_K = 500
PROJECTION_METHOD = 'max'

In [25]:
set_a = alternet_obj21.set_a
set_b  = alternet_obj21.set_b

In [32]:


def score_targets(edges, target_col, importance_col='median_importance'):
    """
    Compute target scores as weighted in-degree.
    score(target) = sum of importance for all edges targeting it
    """
    scores = edges.groupby(target_col)[importance_col].sum()
    return scores.sort_values(ascending=False)



In [36]:
for g in score_targets(plausible_edgesa, 'source_gene', 'S2_median')[0:200].index:
    print(g)

ENSG00000108953
ENSG00000169764
ENSG00000149557
ENSG00000189403
ENSG00000147140
ENSG00000169288
ENSG00000130159
ENSG00000244687
ENSG00000197157
ENSG00000134046
ENSG00000132341
ENSG00000080608
ENSG00000082641
ENSG00000185624
ENSG00000067225
ENSG00000010244
ENSG00000166888
ENSG00000135486
ENSG00000122359
ENSG00000138029
ENSG00000187109
ENSG00000124831
ENSG00000122299
ENSG00000126351
ENSG00000104529
ENSG00000185551
ENSG00000104805
ENSG00000112992
ENSG00000112651
ENSG00000071626
ENSG00000040341
ENSG00000115641
ENSG00000171720
ENSG00000116044
ENSG00000163110
ENSG00000162613
ENSG00000174099
ENSG00000127483
ENSG00000168036
ENSG00000103266
ENSG00000096746
ENSG00000116560
ENSG00000167393
ENSG00000179912
ENSG00000173039
ENSG00000175387
ENSG00000089009
ENSG00000131368
ENSG00000153250
ENSG00000168610
ENSG00000013441
ENSG00000198034
ENSG00000135046
ENSG00000156467
ENSG00000168066
ENSG00000141905
ENSG00000265241
ENSG00000068305
ENSG00000263001
ENSG00000089225
ENSG00000033050
ENSG00000182944
ENSG0000

In [43]:
# for g in sf_edges.sort_values('median_importance', ascending=False)['target_gene'][0:200]:
#     print(g)

for g in score_targets(sf_edges, 'target_gene').index[0:200]:
    print(g)

ENSG00000109971
ENSG00000100650
ENSG00000163660
ENSG00000169045
ENSG00000116754
ENSG00000122566
ENSG00000149257
ENSG00000161960
ENSG00000116560
ENSG00000013441
ENSG00000135486
ENSG00000108654
ENSG00000133112
ENSG00000080824
ENSG00000089280
ENSG00000096746
ENSG00000138668
ENSG00000115541
ENSG00000131051
ENSG00000070756
ENSG00000165119
ENSG00000179094
ENSG00000112081
ENSG00000144381
ENSG00000099622
ENSG00000120694
ENSG00000167978
ENSG00000090621
ENSG00000197111
ENSG00000104904
ENSG00000196531
ENSG00000187514
ENSG00000104824
ENSG00000189403
ENSG00000168066
ENSG00000161547
ENSG00000204628
ENSG00000118194
ENSG00000075415
ENSG00000149273
ENSG00000072778
ENSG00000182944
ENSG00000265681
ENSG00000102317
ENSG00000174444
ENSG00000198563
ENSG00000104852
ENSG00000153187
ENSG00000132341
ENSG00000181163
ENSG00000004478
ENSG00000004534
ENSG00000166710
ENSG00000132424
ENSG00000121310
ENSG00000162613
ENSG00000140400
ENSG00000132716
ENSG00000182149
ENSG00000143569
ENSG00000115310
ENSG00000152795
ENSG0000

In [26]:
plausible_edgesa = set_a[(set_a['is_plausible']) & (set_a.source_category == 'source_isoform_specific')]


In [30]:
plausible_edgesa

,edge_key,source_gene,target_gene,best_tx,S1_mean,S2_mean,S1_median,S2_median,E1,E2,ratio,source_category,max_median,reg_dominance,reg_n_isoforms,is_plausible,filter_reasons,ratio_S2_S1
176359,ENSG00000153406_ENSG00000153406,ENSG00000153406,ENSG00000153406,ENST00000404295,0.0,334.019557,0.0,338.635200,False,True,inf,source_isoform_specific,338.635200,0.634207,2,True,,3.340196e+08
106399,ENSG00000125347_ENSG00000125347,ENSG00000125347,ENSG00000125347,ENST00000405885,0.0,264.621636,0.0,278.384546,False,True,inf,source_isoform_specific,278.384546,0.702679,2,True,,2.646216e+08
3506,ENSG00000125968_ENSG00000125968,ENSG00000125968,ENSG00000125968,ENST00000376112,0.0,252.692664,0.0,259.842351,False,True,inf,source_isoform_specific,259.842351,0.830762,2,True,,2.526927e+08
38596,ENSG00000025434_ENSG00000025434,ENSG00000025434,ENSG00000025434,ENST00000494018,0.0,260.696169,0.0,259.183995,False,True,inf,source_isoform_specific,259.183995,0.550396,2,True,,2.606962e+08
160588,ENSG00000134453_ENSG00000134453,ENSG00000134453,ENSG00000134453,ENST00000447032,0.0,238.086308,0.0,250.286836,False,True,inf,source_isoform_specific,250.286836,0.812849,2,True,,2.380863e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95518,ENSG00000143742_ENSG00000175768,ENSG00000143742,ENSG00000175768,ENST00000304786,0.0,3.936189,0.0,3.350289,False,True,inf,source_isoform_specific,3.350289,0.907830,2,True,,3.936189e+06
142287,ENSG00000124440_ENSG00000115290,ENSG00000124440,ENSG00000115290,ENST00000377670,0.0,3.308679,0.0,3.350272,False,True,inf,source_isoform_specific,3.350272,0.462791,5,True,,3.308679e+06
190393,ENSG00000104852_ENSG00000141759,ENSG00000104852,ENSG00000141759,ENST00000598441,0.0,3.367764,0.0,3.350256,False,True,inf,source_isoform_specific,3.350256,0.357750,7,True,,3.367764e+06
47941,ENSG00000111669_ENSG00000221983,ENSG00000111669,ENSG00000221983,ENST00000396705,0.0,3.552067,0.0,3.350229,False,True,inf,source_isoform_specific,3.350229,0.921956,4,True,,3.552067e+06


In [31]:
for g in plausible_edgesa['source_gene'][0:200]:
    print(g)

ENSG00000153406
ENSG00000125347
ENSG00000125968
ENSG00000025434
ENSG00000134453
ENSG00000065534
ENSG00000116132
ENSG00000120837
ENSG00000166925
ENSG00000105516
ENSG00000164104
ENSG00000177508
ENSG00000100968
ENSG00000198517
ENSG00000066336
ENSG00000170315
ENSG00000121691
ENSG00000124440
ENSG00000198911
ENSG00000072310
ENSG00000177426
ENSG00000122034
ENSG00000196843
ENSG00000168874
ENSG00000126254
ENSG00000168610
ENSG00000114315
ENSG00000109906
ENSG00000170581
ENSG00000183072
ENSG00000095794
ENSG00000170485
ENSG00000136574
ENSG00000135439
ENSG00000146592
ENSG00000134046
ENSG00000172922
ENSG00000117595
ENSG00000105856
ENSG00000162772
ENSG00000109819
ENSG00000107281
ENSG00000136997
ENSG00000025156
ENSG00000180353
ENSG00000188290
ENSG00000100219
ENSG00000108788
ENSG00000137834
ENSG00000111328
ENSG00000106628
ENSG00000116016
ENSG00000130522
ENSG00000069399
ENSG00000067082
ENSG00000100811
ENSG00000136826
ENSG00000175197
ENSG00000157557
ENSG00000198034
ENSG00000173153
ENSG00000131368
ENSG0000

In [ ]:
# 1. Filter for plausible edges and sort them
plausible_edges = set_b[(set_b['is_plausible']) & (set_b.target_category == 'target_isoform_specific')].sort_values('max_mean', ascending=False)




In [ ]:
for g in sf_edges['source_transcript'].value_counts().sort_values( ascending=False)[0:200].index:
    print(g)

ENST00000602219
ENST00000470824
ENST00000595443
ENST00000627714
ENST00000604367
ENST00000605057
ENST00000463920
ENST00000591991
ENST00000504045
ENST00000323853
ENST00000601047
ENST00000614136
ENST00000463328
ENST00000513230
ENST00000553501
ENST00000586778
ENST00000519943
ENST00000469838
ENST00000470781
ENST00000338631
ENST00000479275
ENST00000366527
ENST00000513972
ENST00000375650
ENST00000485841
ENST00000438552
ENST00000512903
ENST00000554465
ENST00000463877
ENST00000375651
ENST00000581979
ENST00000464087
ENST00000532683
ENST00000464988
ENST00000487045
ENST00000551098
ENST00000310513
ENST00000513036
ENST00000467302
ENST00000457156
ENST00000409406
ENST00000593845
ENST00000496183
ENST00000463116
ENST00000573681
ENST00000554680
ENST00000532091
ENST00000376256
ENST00000534279
ENST00000592599
ENST00000421682
ENST00000558036
ENST00000381793
ENST00000484270
ENST00000504643
ENST00000484162
ENST00000483853
ENST00000579996
ENST00000496179
ENST00000464316
ENST00000607145
ENST00000528062
ENST0000

In [ ]:
for g in sf_edges.sort_values('median_importance', ascending=False)['target_gene'][0:200]:
    print(g)

In [ ]:
sf_edges['reg_type'] = 'SF'

In [ ]:
regs = pd.concat([plausible_edges.copy(), sf_edges.copy()])

In [ ]:
filtered_edges = regs.groupby('target_gene').filter(
    lambda x: ('SF' in x['reg_type'].values) and ('TF' in x['reg_type'].values)
)

In [ ]:
for g in filtered_edges.sort_values('S3_mean')['target_gene'][0:150]:
    print(g)

ENSG00000197756
ENSG00000139438
ENSG00000119699
ENSG00000188846
ENSG00000089335
ENSG00000113916
ENSG00000142871
ENSG00000054967
ENSG00000169246
ENSG00000189190
ENSG00000160593
ENSG00000151914
ENSG00000198373
ENSG00000072778
ENSG00000149925
ENSG00000167978
ENSG00000183828
ENSG00000173267
ENSG00000125447
ENSG00000132024
ENSG00000133943
ENSG00000005379
ENSG00000146674
ENSG00000198668
ENSG00000153339
ENSG00000073921
ENSG00000196684
ENSG00000055163
ENSG00000138772
ENSG00000012124
ENSG00000005844
ENSG00000141556
ENSG00000132475
ENSG00000135218
ENSG00000117616
ENSG00000154134
ENSG00000115468
ENSG00000090924
ENSG00000126456
ENSG00000172037
ENSG00000105483
ENSG00000143537
ENSG00000087266
ENSG00000130725
ENSG00000129467
ENSG00000175215
ENSG00000109846
ENSG00000163636
ENSG00000080824
ENSG00000100731
ENSG00000059377
ENSG00000105701
ENSG00000122966
ENSG00000141497
ENSG00000266714
ENSG00000198467
ENSG00000205436
ENSG00000043462
ENSG00000158023
ENSG00000135218
ENSG00000177575
ENSG00000128335
ENSG0000

In [ ]:
for g in filtered_edges['target_gene'][0:100]:
    print(g)

In [ ]:
plausible_edges['source_gene'] =   plausible_edges['source_transcript'].map(tx2gene)


In [ ]:
plausible_edges[(plausible_edges.source_transcript ==  'ENST00000509339') & (plausible_edges.target_gene == 'ENSG00000197249')]

In [ ]:
plausible_edges[plausible_edges.target_gene == 'ENSG00000266714']['source_gene'].value_counts()

In [ ]:
for g in plausible_edges.drop_duplicates(subset=['target_gene', 'source_gene'])['target_gene'].value_counts().reset_index(name='unique_incoming_edges').head(50)['target_gene']:
    print(g)

In [ ]:
for g in plausible_edges.drop_duplicates(subset=['target_gene', 'source_gene'])['target_gene'].value_counts().reset_index(name='unique_incoming_edges').head(50)['target_gene']:
    print(g)

In [ ]:
# 1. Filter for plausible, isoform-specific rows
filtered_df = set_a[(set_a["is_plausible"]) & (set_a["source_category"] == "source_isoform_specific")]

# 2. Group by target and calculate the in-degree (unique incoming source genes)
target_in_degree = (
    filtered_df.groupby("target_gene")["source_gene"]
    .nunique()
    .reset_index(name="in_degree")
)

# 3. Sort by in-degree in descending order and slice the top 200
top_targets = target_in_degree.sort_values("in_degree", ascending=False).head(200)

# 4. Print the top target genes
for g in top_targets["target_gene"]:
    print(g)

In [ ]:
set_b

In [ ]:
# Filter Set B for target_isoform_unique edges
l3_table2 = set_b[(set_b['target_category'] == 'target_isoform_specific') & (set_b['is_plausible'])].copy()

l3_top, l3_symbols, l3_meta = build_target_list(
    edges=l3_table2,
    target_col='target_transcript',
    importance_col='S3_mean',
    target_type='transcript',
    tx2gene=tx2gene,
    gene2symbol=gene2symbol,
    K=250,
    projection_method=PROJECTION_METHOD
)



In [ ]:
for g in l3_top['gene_symbol']:
    print(g)